# Swiss Legal Citation Retrieval — Interactive Demo

Paste an English case description and get the top-ranked **Swiss legal citations** grounded against the corpus.

**How it works:**
1. `llama3.1:8b` translates your English query to German/French/Italian
2. BM25 retrieves relevant statute chunks from the German laws corpus
3. `llama3.1:8b` reranks the candidate citations by relevance to your question
4. Results are emitted in abbreviation form (e.g. `Art. 97 Abs. 1 OR`) — the form used in Swiss legal citations

**Prereqs:**
1. `pip install -r requirements.txt`
2. Download data (see `README.md`) into `data/`
3. Build the index once: `python index_builder.py`
4. Start Ollama: `ollama serve` (in a separate terminal)
5. Pull the model: `ollama pull llama3.1:8b`

## 1. Load the retriever

In [ ]:
from Law_evaluation import LegalRetriever

# llama3.2:3b for both translation and reranking (fast enough for long legal prompts)
retriever = LegalRetriever(model="llama3.2:3b", translate_model="llama3.2:3b")
retriever.load()
print(f"Indexed citations: {len(retriever._valid_citations):,}")

## 2. Ask a legal question

Replace `case` with your own English case description.
The pipeline will translate it, retrieve candidates, and rerank them with `llama3.1:8b`.

In [ ]:
case = (
    "My client bought a used car in Zurich. Two weeks later the engine failed "
    "and a mechanic found a cracked block that must have existed before the sale. "
    "The seller refuses a refund. What Swiss legal basis can we invoke?"
)

citations = retriever.predict(case)

print(f"Found {len(citations)} citations:\n")
for c in citations:
    print(f"  {c}")

## 3. Inspect the source snippets for each citation

In [ ]:
from citation_norm import ABBREV_TO_SR

chunks = retriever._chunks
for c in citations:
    # Corpus stores SR-numeric form; try both abbrev and numeric.
    abbr_tail = c.split()[-1]
    sr = ABBREV_TO_SR.get(abbr_tail, "")
    sr_key = c.replace(abbr_tail, sr).strip() if sr else c
    hits = chunks[chunks["citation"].isin([c, sr_key])]
    if hits.empty:
        continue
    print(f"[{c}]")
    print(str(hits.iloc[0]["text"])[:400])
    print("-" * 60)

## 4. Score against the validation split

Runs the full LLM pipeline over `val.csv` and scores with Macro F1.
This takes ~15 minutes (10 queries × 3 LLM calls each).

In [ ]:
val_df = retriever.run_on_split("val")
val_df.to_csv("val_submission.csv", index=False)
val_df.head()

In [ ]:
!python evaluation/evaluate.py val_submission.csv --split val -v

## 5. Generate the test submission

~60 minutes for 40 queries (3 LLM calls each at ~1 min per call).

In [ ]:
test_df = retriever.run_on_split("test")
test_df.to_csv("submission.csv", index=False)
print(f"Wrote {len(test_df)} predictions to submission.csv")
test_df.head()